In [ ]:
import os
import torch
import gzip
import subprocess
import logging
import pandas as pd
import numpy as np
import biotite.structure.io.pdbx as pdbx
from typing import Dict, List, Tuple

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ClusterSplitTracker:
    """Tracks sequence extraction and clustering metrics."""
    def __init__(self):
        self.extracted_sequences = 0
        self.extraction_failed = []
        self.total_clusters = 0
        self.train_proteins = 0
        self.test_proteins = 0

    def print_summary(self):
        print("\n" + "="*50)
        print("🧬 SEQUENCE CLUSTERING & SPLIT SUMMARY")
        print("="*50)
        print(f"Sequences Extracted:        {self.extracted_sequences}")
        print(f"Extraction Failures:        {len(self.extraction_failed)}")
        print(f"Total Unique Clusters:      {self.total_clusters}")
        print(f"Proteins assigned to Train: {self.train_proteins} (approx 80% of clusters)")
        print(f"Proteins assigned to Test:  {self.test_proteins}  (approx 20% of clusters)")
        print("="*50 + "\n")

class SequenceClusterSplitter:
    def __init__(self, cif_dir: str = "alphafold_cif", work_dir: str = "clustering_work"):
        self.cif_dir = cif_dir
        self.work_dir = work_dir
        self.fasta_path = os.path.join(self.work_dir, "all_sequences.fasta")
        self.cluster_out_prefix = os.path.join(self.work_dir, "cluster_res")
        self.tracker = ClusterSplitTracker()
        
        if not os.path.exists(self.work_dir):
            os.makedirs(self.work_dir)

    def extract_sequences(self, uniprot_ids: set) -> Dict[str, str]:
        """Reads .cif.gz files using Biotite and extracts the 1D amino acid sequence."""
        logger.info(f"Extracting sequences from {len(uniprot_ids)} CIF files...")
        seq_dict = {}
        
        for i, uid in enumerate(uniprot_ids):
            if i > 0 and i % 1000 == 0:
                logger.info(f"Extracted {i}/{len(uniprot_ids)} sequences...")
                
            file_path = os.path.join(self.cif_dir, f"AF-{uid}.cif.gz")
            if not os.path.exists(file_path):
                # Fallback to v4 naming convention if necessary
                file_path = os.path.join(self.cif_dir, f"AF-{uid}-F1-model_v4.cif.gz")
                
            if not os.path.exists(file_path):
                self.tracker.extraction_failed.append((uid, "File missing"))
                continue

            try:
                # FIX 1: Use CIFFile instead of the deprecated PDBxFile
                with gzip.open(file_path, "rt") as f:
                    cif_file = pdbx.CIFFile.read(f)
                    
                # FIX 2: get_sequence now returns a dict mapping chain IDs to sequences
                sequence_dict = pdbx.get_sequence(cif_file)
                
                if sequence_dict:
                    # AlphaFold models typically have one chain, so we extract the first value
                    seq = str(list(sequence_dict.values())[0])
                    seq_dict[uid] = seq
                    self.tracker.extracted_sequences += 1
                else:
                    self.tracker.extraction_failed.append((uid, "No sequence found in CIF"))
            except Exception as e:
                self.tracker.extraction_failed.append((uid, str(e)))

        # Save to FASTA
        logger.info("Writing sequences to FASTA file...")
        with open(self.fasta_path, "w") as f:
            for uid, seq in seq_dict.items():
                f.write(f">{uid}\n{seq}\n")
                
        return seq_dict

    def run_mmseqs_clustering(self) -> str:
        """Executes MMseqs2 locally to cluster sequences at 30% identity."""
        logger.info("Starting MMseqs2 clustering (this may take a few minutes)...")
        tmp_dir = os.path.join(self.work_dir, "tmp")
        os.makedirs(tmp_dir, exist_ok=True)

        # Command: mmseqs easy-cluster [in.fasta] [out_prefix] [tmp_dir] --min-seq-id 0.3 -c 0.8
        # -c 0.8 means 80% coverage required (standard for domain-level clustering)
        cmd = [
            "mmseqs", "easy-cluster", 
            self.fasta_path, self.cluster_out_prefix, tmp_dir,
            "--min-seq-id", "0.3", 
            "-c", "0.8", 
            "--cov-mode", "1"
        ]

        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            logger.info("MMseqs2 clustering completed successfully.")
        except subprocess.CalledProcessError as e:
            logger.error(f"MMseqs2 failed. Error:\n{e.stderr.decode()}")
            print(self.tracker.extraction_failed)
            raise e
            
        return f"{self.cluster_out_prefix}_cluster.tsv"

    def create_split_mapping(self, cluster_tsv: str, train_ratio: float = 0.8) -> Dict[str, str]:
        """
        Parses the MMseqs2 output TSV and assigns entire clusters to Train or Test.
        Returns a dict mapping UniProt ID -> 'train' or 'test'.
        """
        logger.info("Parsing clusters and assigning train/test splits...")
        
        # MMseqs2 TSV format: Column 1 = Cluster Representative, Column 2 = Cluster Member
        df_clusters = pd.read_csv(cluster_tsv, sep='\t', header=None, names=['Rep', 'Member'])
        
        # Group members by their representative
        clusters = df_clusters.groupby('Rep')['Member'].apply(list).to_dict()
        self.tracker.total_clusters = len(clusters)
        
        # Shuffle clusters for random assignment, but keep it deterministic with a seed
        cluster_reps = list(clusters.keys())
        np.random.seed(42)
        np.random.shuffle(cluster_reps)
        
        split_idx = int(len(cluster_reps) * train_ratio)
        train_clusters = cluster_reps[:split_idx]
        test_clusters = cluster_reps[split_idx:]
        
        uid_to_split = {}
        
        for rep in train_clusters:
            for member in clusters[rep]:
                uid_to_split[member] = 'train'
                self.tracker.train_proteins += 1
                
        for rep in test_clusters:
            for member in clusters[rep]:
                uid_to_split[member] = 'test'
                self.tracker.test_proteins += 1

        return uid_to_split


def run_phase_5(df: pd.DataFrame) -> pd.DataFrame:
    """Orchestrates the sequence extraction, clustering, and dataframe updating."""
    logger.info("Starting Phase 5: 30% Sequence Identity Split...")
    
    unique_uniprot = set(df['UniProt_ID'].dropna().unique())
    splitter = SequenceClusterSplitter(cif_dir="alphafold_cif")
    
    # 1. Extract sequences and save to FASTA
    splitter.extract_sequences(unique_uniprot)
    
    # 2. Run MMseqs2 clustering
    cluster_tsv = splitter.run_mmseqs_clustering()
    
    # 3. Create the Train/Test mapping at the cluster level
    uid_to_split = splitter.create_split_mapping(cluster_tsv)
     
    # 4. Map the split back to the massive manifest DataFrame
    df['Dataset_Split'] = df['UniProt_ID'].map(uid_to_split)
    
    # Drop rows where extraction failed (if any)
    initial_len = len(df)
    df_filtered = df.dropna(subset=['Dataset_Split']).copy()
    
    splitter.tracker.print_summary()
    logger.info(f"Dropped {initial_len - len(df_filtered)} rows due to sequence extraction failures.")
    
    return df_filtered, splitter.tracker

# Example Execution Hook:
# final_df_phase4 = pd.read_csv("phase4_structural_manifest.csv")
# final_df_phase5 = run_phase_5(final_df_phase4)
# final_df_phase5.to_csv("phase5_split_manifest.csv", index=False)

In [ ]:
df = pd.read_csv("step_2_with_smiles.csv", index_col = 0)

In [ ]:
filtered_df, tracker = run_phase_5(df)

In [ ]:
filtered_df["Dataset_Split"].value_counts()

In [ ]:
filtered_df.to_csv("train_test.csv")

## checking

In [ ]:
import os

In [ ]:
df = pd.read_csv("train_test.csv", index_col = 0)

In [ ]:
df.head(2)

In [ ]:
lst = []
for row in df[["Valid_SMILES", "UniProt_ID",]].iterrows():
    row = row[1]
    smiles, protein = eval(row.Valid_SMILES), row.UniProt_ID
    lst.append(int(os.path.exists(f"alphafold_cif/AF-{protein}.cif.gz")) * len(smiles))

In [ ]:
lst[:10]

In [ ]:
ligands = set()
for ele in df["Valid_SMILES"].apply(eval):
    for ele2 in ele:
        ligands.add(ele2)

In [ ]:
len(ligands)

In [ ]:
df.shape

In [ ]:
df["UniProt_ID"].nunique()

In [ ]:
sum(lst)

In [ ]:
df.shape

In [ ]:
import glob
x = glob.glob("alphafold_cif/*")

In [ ]:
len(x)

In [ ]:
x[0]

In [ ]:
df.head(1).to_dict()

#### new approach

In [ ]:
from tqdm import tqdm

In [ ]:
import pandas as pd
import ast
import random
import os
import gzip
import shutil
import subprocess
import csv
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina

# ==========================================
# 0. The Pipeline Tracker
# ==========================================
class PipelineTracker:
    def __init__(self, log_file='pipeline_errors.csv'):
        self.log_file = log_file
        self.errors = []
        
    def log(self, entity_id, stage, reason):
        """Logs an error for a specific entity (UniProt ID or SMILES)."""
        self.errors.append({
            'Entity_ID': entity_id,
            'Stage': stage,
            'Reason': reason
        })
        
    def export(self):
        """Saves all tracked errors to a CSV."""
        if not self.errors:
            print("Tracker: No errors recorded! Clean run.")
            return
            
        keys = self.errors[0].keys()
        with open(self.log_file, 'w', newline='') as f:
            dict_writer = csv.DictWriter(f, fieldnames=keys)
            dict_writer.writeheader()
            dict_writer.writerows(self.errors)
        print(f"Tracker: {len(self.errors)} errors logged to {self.log_file}")

# ==========================================
# 1. Sequence Extraction & Protein Clustering
# ==========================================
def extract_sequence_from_cif(cif_gz_path, uid, tracker):
    """Extracts the 1D amino acid sequence from an AlphaFold .cif.gz file."""
    seq = []
    try:
        with gzip.open(cif_gz_path, 'rt') as f:
            capture = False
            for line in f:
                if line.startswith('_entity_poly.pdbx_seq_one_letter_code_can'):
                    parts = line.split()
                    if len(parts) > 1:
                        return parts[1].strip(';')
                    capture = True
                elif capture:
                    if line.startswith(';'):
                        if seq: break
                        seq.append(line.strip('; \n'))
                    elif line.startswith('_'):
                        break
                    else:
                        seq.append(line.strip())
            
            final_seq = "".join(seq).replace('\n', '').replace(' ', '')
            if not final_seq:
                tracker.log(uid, 'Sequence Extraction', 'Sequence tag found but empty')
            return final_seq
            
    except Exception as e:
        tracker.log(uid, 'Sequence Extraction', f'File Read Error: {str(e)}')
        return None

def cluster_proteins_mmseqs(df, tracker, cif_dir='alphafold_cif', using_alphafold=False, uniprot_to_seq = dict(), identity=0.3):
    fasta_path = 'temp_proteins.fasta'
    db_path = 'temp_db'
    cluster_path = 'temp_clu'
    tsv_path = 'temp_clu.tsv'
    
    unique_uniprot = df['UniProt_ID'].unique()
    print(f"Extracting sequences for {len(unique_uniprot)} unique proteins...")
    
    valid_proteins = 0
    with open(fasta_path, 'w') as f:
        for uid in unique_uniprot:
            if using_alphafold:
                cif_path = os.path.join(cif_dir, f"AF-{uid}.cif.gz")
                if not os.path.exists(cif_path):
                    tracker.log(uid, 'File Check', 'CIF file not found')
                    continue

                seq = extract_sequence_from_cif(cif_path, uid, tracker)
            else:
                try:
                    seq = uniprot_to_seq[uid]
                except KeyError:
                    tracker.log(uid, 'File Check', 'UID not present in dict')

            if seq:
                f.write(f">{uid}\n{seq}\n")
                valid_proteins += 1
                
    if valid_proteins == 0:
        raise ValueError("Critical Failure: No valid protein sequences extracted. Check CIF directory.")
    
    print("Running MMseqs2 clustering at 30% identity...")
    subprocess.run(['mmseqs', 'createdb', fasta_path, db_path], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['mmseqs', 'cluster', db_path, cluster_path, 'temp_tmp', '--min-seq-id', str(identity)], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['mmseqs', 'createtsv', db_path, db_path, cluster_path, tsv_path], check=True, stdout=subprocess.DEVNULL)
    
    print("MMSeq run successful")
    protein_to_cluster = {}
    with open(tsv_path, 'r') as f:
        for line in f:
            rep, member = line.strip().split('\t')
            protein_to_cluster[member] = rep
            
    # Cleanup

    for ext in ['', '.index', '.dbtype', '_h', '_h.index', '_h.dbtype']:
        for base in [db_path, cluster_path]:
            if os.path.exists(base+ext): os.remove(base+ext)
    for tmp_file in [fasta_path, tsv_path]:
        if os.path.exists(tmp_file): os.remove(tmp_file)

    # --- CHANGED LINE HERE ---
    if os.path.exists('temp_tmp'): 
        shutil.rmtree('temp_tmp') 

    return protein_to_cluster


# ==========================================
# 2. SMILES Extraction & Ligand Clustering
# ==========================================
def cluster_ligands_butina(df, tracker, smiles_col='Valid_SMILES', similarity_cutoff=0.4):
    print("butina called")
    smiles_list = df[smiles_col].unique()
#     for row in df[smiles_col]:
#         smiles_list = ast.literal_eval(row) if isinstance(row, str) else row
#         for s in smiles_list:
#             unique_smiles.add(s)
            
#     smiles_list = list(unique_smiles)
    print(f"Clustering {len(smiles_list)} unique ligands...")
    
    valid_smiles = []
    mols = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if m is not None:
            mols.append(m)
            valid_smiles.append(s)
        else:
            tracker.log(s, 'RDKit Parsing', 'Invalid SMILES string')
            
    if not mols:
         raise ValueError("Critical Failure: No valid SMILES could be parsed by RDKit.")
    
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols]
    
    dists = []
    nfps = len(fps)
    for i in range(1, nfps):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1 - x for x in sims])
        
    clusters = Butina.ClusterData(dists, nfps, distThresh=(1.0 - similarity_cutoff), isDistData=True)
    
    smiles_to_cluster = {}
    for cluster_id, cluster_indices in enumerate(clusters):
        for idx in cluster_indices:
            smiles_to_cluster[valid_smiles[idx]] = f"L_CLU_{cluster_id}"
            
    return smiles_to_cluster

# ==========================================
# 3. Bipartite Grid Optimization
# ==========================================
def optimize_bipartite_split(edges, p_clusters, l_clusters, iterations=20000):
    print("Optimizing grid assignment to hit target quotas...")
    
    p_split = {p: ('train' if random.random() < 0.85 else 'test') for p in p_clusters}
    l_split = {l: ('train' if random.random() < 0.85 else 'test') for l in l_clusters}
    
    targets = {'strict_train': 0.70, 'cold_target': 0.10, 'cold_drug': 0.10, 'strict_test': 0.10}
    total_edges = sum(edges.values())
    
    def calculate_loss(p_split_map, l_split_map):
        quadrants = defaultdict(int)
        for (p, l), count in edges.items():
            ps, ls = p_split_map[p], l_split_map[l]
            if ps == 'train' and ls == 'train': quadrants['strict_train'] += count
            elif ps == 'test' and ls == 'train': quadrants['cold_target'] += count
            elif ps == 'train' and ls == 'test': quadrants['cold_drug'] += count
            else: quadrants['strict_test'] += count
            
        loss = sum(abs((quadrants[q] / total_edges) - target_pct) for q, target_pct in targets.items())
        return loss, quadrants
        
    current_loss, current_quads = calculate_loss(p_split, l_split)
    all_nodes = [('p', p) for p in p_clusters] + [('l', l) for l in l_clusters]
    
    for _ in range(iterations):
        node_type, node = random.choice(all_nodes)
        if node_type == 'p':
            old_val = p_split[node]
            p_split[node] = 'test' if old_val == 'train' else 'train'
            new_loss, new_quads = calculate_loss(p_split, l_split)
            if new_loss < current_loss:
                current_loss, current_quads = new_loss, new_quads
            else:
                p_split[node] = old_val 
        else:
            old_val = l_split[node]
            l_split[node] = 'test' if old_val == 'train' else 'train'
            new_loss, new_quads = calculate_loss(p_split, l_split)
            if new_loss < current_loss:
                current_loss, current_quads = new_loss, new_quads
            else:
                l_split[node] = old_val 

    print("\nOptimized Split Distribution (Exploded Edges):")
    for q, count in current_quads.items():
        print(f"  {q}: {count/total_edges*100:.2f}% ({count} pairs)")
        
    return p_split, l_split

# ==========================================
# 4. Main Execution Pipeline
# ==========================================
def build_dataset_split(df, uniprot_to_seq = dict(), using_alphafold =False, cif_dir='alphafold_cif'):
    try:
        tracker = PipelineTracker()

        protein_to_cluster = cluster_proteins_mmseqs(df, tracker, cif_dir, uniprot_to_seq=uniprot_to_seq, using_alphafold=False)
        print("protein clustered")
        smiles_to_cluster = cluster_ligands_butina(df, tracker)

        print("\nExploding dataframe based on ligands...")
#         records = []
#         for _, row in df.iterrows():
#             smiles_list = ast.literal_eval(row['Valid_SMILES']) if isinstance(row['Valid_SMILES'], str) else row['Valid_SMILES']
#             for smile in smiles_list:
#                 new_row = row.copy()
#                 new_row['Single_SMILES'] = smile
#                 records.append(new_row)

#         exploded_df = pd.DataFrame(records)
        exploded_df = df
        print(f"Original rows: {len(df)} -> Exploded interactions: {len(exploded_df)}")

        exploded_df['P_Cluster'] = exploded_df['UniProt_ID'].map(protein_to_cluster)
        exploded_df['L_Cluster'] = exploded_df['Valid_SMILES'].map(smiles_to_cluster)

        # Track items dropped due to clustering failures
        dropped_df = exploded_df[exploded_df['P_Cluster'].isna() | exploded_df['L_Cluster'].isna()]
        for _, row in dropped_df.iterrows():
            if pd.isna(row['P_Cluster']):
                tracker.log(row['UniProt_ID'], 'Merge', 'Dropped due to missing protein cluster')
            if pd.isna(row['L_Cluster']):
                tracker.log(row['Single_SMILES'], 'Merge', 'Dropped due to missing ligand cluster')

        exploded_df = exploded_df.dropna(subset=['P_Cluster', 'L_Cluster'])

        edges = exploded_df.groupby(['P_Cluster', 'L_Cluster']).size().to_dict()
        p_clusters = exploded_df['P_Cluster'].unique()
        l_clusters = exploded_df['L_Cluster'].unique()

        p_split, l_split = optimize_bipartite_split(edges, p_clusters, l_clusters)

        def get_quadrant(row):
            ps = p_split[row['P_Cluster']]
            ls = l_split[row['L_Cluster']]
            if ps == 'train' and ls == 'train': return 'strict_train'
            elif ps == 'test' and ls == 'train': return 'cold_target'
            elif ps == 'train' and ls == 'test': return 'cold_drug'
            else: return 'strict_test'

        exploded_df['Dataset_Split'] = exploded_df.apply(get_quadrant, axis=1)

        tracker.export()
    except Exception as e:
        print(e)
        return pd.DataFrame(), tracker
    return exploded_df, tracker

# --- Usage Example ---
# df = pd.read_csv('your_dataset.csv')
# final_exploded_df = build_dataset_split(df, cif_dir='./alphafold_cif')
# final_exploded_df.to_csv('ready_for_training.csv', index=False)

In [ ]:
df = pd.read_parquet("dataset_164_step2.pq")

In [ ]:
df.head(1)

In [ ]:
uniprot_to_seq = torch.load("uniprot_to_sequence_49700.pt")

In [ ]:
# df["Ligands"] = df["Ligands"].apply(eval)
# df["Valid_SMILES"] = df["Valid_SMILES"].apply(eval)

In [ ]:
exploded_df, tracker = build_dataset_split(df, uniprot_to_seq=uniprot_to_seq)

In [ ]:
exploded_df.to_parquet("dataset_164_step4.pq")

In [ ]:
uniprot_to_seq["Q7A3D8"]

In [ ]:
tracker

In [ ]:
reasons = dict()
for e in tracker.errors:
    reasons[e["Reason"]] = reasons.get(e["Reason"], 0) + 1

In [ ]:
reasons

In [ ]:
exploded_df.to_parquet("train_test_split_hard_cold.pq")

In [ ]:
for e in tracker.errors:
    if e["Reason"] == "Dropped due to missing protein cluster":
        break

In [ ]:
exploded_df["Dataset_Split"].value_counts(normalize=True).round(2) * 100

In [ ]:
exploded_df.shape